In [30]:
import os
import re
import requests
import time
import pandas as pd

NVD_API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"


def extract_cpe_matches(value):
    """Extract CPE match entries recursively from NVD configurations."""
    if isinstance(value, dict):
        for match in value.get("cpeMatch", []):
            yield match
        for key, child in value.items():
            if key != "cpeMatch":
                yield from extract_cpe_matches(child)
    elif isinstance(value, list):
        for child in value:
            yield from extract_cpe_matches(child)


def fetch_cves(
    keyword, results_per_page=100, start_date=None, end_date=None, max_results=None
):
    """Fetch CVEs with retries, pagination, and optional last-modified filtering."""
    if not 1 <= results_per_page <= 2000:
        raise ValueError("results_per_page must be between 1 and 2000")
    if max_results is None:
        max_results = results_per_page
    if max_results < 1:
        raise ValueError("max_results must be positive")

    start = pd.to_datetime(start_date, utc=True) if start_date is not None else None
    end = pd.to_datetime(end_date, utc=True) if end_date is not None else None
    end_is_date = isinstance(end_date, str) and len(end_date) == 10

    if start is not None and end is not None and (
        start >= end + pd.Timedelta(days=1) if end_is_date else start > end
    ):
        raise ValueError("start_date must be before or equal to end_date")

    headers = {}
    api_key = os.getenv("NVD_API_KEY")
    if api_key:
        headers["apiKey"] = api_key

    records = []
    total_results = 0
    fetched_count = 0
    page_size = results_per_page
    while fetched_count < max_results:
        page_size = min(page_size, max_results - fetched_count)
        retry_count = 0
        while True:
            try:
                response = requests.get(
                    NVD_API_URL,
                    params={
                        "keywordSearch": keyword,
                        "resultsPerPage": page_size,
                        "startIndex": fetched_count,
                    },
                    headers=headers,
                    timeout=(10, 60),
                )
                if response.status_code == 429 or response.status_code >= 500:
                    if retry_count >= 4:
                        response.raise_for_status()
                    retry_after = response.headers.get("Retry-After")
                    try:
                        delay = float(retry_after) if retry_after else 2 ** retry_count
                    except ValueError:
                        delay = 2 ** retry_count
                    time.sleep(min(delay, 60))
                    retry_count += 1
                    continue
                response.raise_for_status()
                payload = response.json()
                break
            except requests.exceptions.ChunkedEncodingError:
                if page_size > 1:
                    page_size = max(1, page_size // 2)
                    retry_count = 0
                    continue
                retry_count += 1
                if retry_count >= 4:
                    raise
                time.sleep(2 ** retry_count)
            except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
                retry_count += 1
                if retry_count >= 4:
                    raise
                time.sleep(2 ** retry_count)

        if fetched_count == 0:
            total_results = payload.get("totalResults", 0)
        page_items = payload.get("vulnerabilities", [])
        if not page_items:
            break
        for item in page_items:
            cve = item["cve"]
            descriptions = cve.get("descriptions", [])
            description = next(
                (d["value"] for d in descriptions if d.get("lang") == "en"), ""
            )

            metrics = cve.get("metrics", {})
            cvss = None
            cvss_version = None
            severity = None
            attack_vector = None
            exploitability_score = None
            for metric_name in (
                "cvssMetricV40", "cvssMetricV31", "cvssMetricV30", "cvssMetricV2"
            ):
                entries = metrics.get(metric_name, [])
                if entries:
                    data = entries[0].get("cvssData", {})
                    cvss = data.get("baseScore")
                    cvss_version = {
                        "cvssMetricV40": "4.0",
                        "cvssMetricV31": "3.1",
                        "cvssMetricV30": "3.0",
                        "cvssMetricV2": "2.0",
                    }[metric_name]
                    severity = data.get("baseSeverity") or entries[0].get("baseSeverity")
                    attack_vector = data.get("attackVector") or data.get("accessVector")
                    exploitability_score = entries[0].get("exploitabilityScore")
                    break

            cwe_ids = sorted({
                weakness_description.get("value")
                for weakness in cve.get("weaknesses", [])
                for weakness_description in weakness.get("description", [])
                if weakness_description.get("lang") == "en"
                and (weakness_description.get("value") or "").startswith("CWE-")
            })

            affected_products = set()
            for match in extract_cpe_matches(cve.get("configurations", [])):
                criteria = match.get("criteria", "")
                cpe_parts = criteria.split(":")
                if match.get("vulnerable") and len(cpe_parts) > 4:
                    affected_products.add(f"{cpe_parts[3]}:{cpe_parts[4]}")

            references = [
                ref.get("url")
                for ref in cve.get("references", [])
                if ref.get("url")
            ]
            related_cves = sorted({
                related_id
                for related_id in re.findall(
                    r"CVE-\d{4}-\d{4,}",
                    " ".join([description, *references]),
                    flags=re.IGNORECASE,
                )
                if related_id.upper() != (cve.get("id") or "").upper()
            })

            records.append({
                "cve_id": cve.get("id"),
                "published": cve.get("published"),
                "last_modified": cve.get("lastModified"),
                "description": description,
                "cvss_score": cvss,
                "cvss_version": cvss_version,
                "severity": severity,
                "attack_vector": attack_vector,
                "exploitability_score": exploitability_score,
                "cwe_ids": cwe_ids,
                "affected_products": sorted(affected_products),
                "references": references,
                "related_cves": related_cves,
            })
        fetched_count += len(page_items)
        if fetched_count >= total_results or fetched_count >= max_results:
            break
        if not api_key:
            time.sleep(6)

    df = pd.DataFrame(records, columns=[
        "cve_id", "published", "last_modified", "description", "cvss_score",
        "cvss_version",
        "severity", "attack_vector", "exploitability_score", "cwe_ids",
        "affected_products", "references", "related_cves",
    ])
    if not df.empty and (start is not None or end is not None):
        last_modified = pd.to_datetime(df["last_modified"], utc=True)
        mask = pd.Series(True, index=df.index)
        if start is not None:
            mask &= last_modified >= start
        if end is not None:
            if end_is_date:
                mask &= last_modified < end + pd.Timedelta(days=1)
            else:
                mask &= last_modified <= end
        df = df.loc[mask].reset_index(drop=True)

    df.attrs["nvd_total_results"] = total_results
    df.attrs["nvd_page_size"] = page_size
    df.attrs["nvd_fetched_results"] = fetched_count
    return df



In [31]:
df = fetch_cves("Apache")

In [32]:
def fetch_cves_for_keywords(
    keywords, results_per_page=100, start_date=None, end_date=None, max_results=None
):
    """Busca CVEs para várias keywords e remove resultados duplicados."""
    if isinstance(keywords, str):
        keywords = [keywords]

    frames = [
        fetch_cves(
            keyword,
            results_per_page=results_per_page,
            start_date=start_date,
            end_date=end_date,
            max_results=max_results,
        )
        for keyword in keywords
    ]

    frames = [frame for frame in frames if not frame.empty]
    if not frames:
        return pd.DataFrame()

    return (
        pd.concat(frames, ignore_index=True)
        .drop_duplicates(subset="cve_id")
        .reset_index(drop=True)
    )

In [33]:
fetch_cves_for_keywords("%")

,cve_id,published,last_modified,description,cvss_score,cvss_version,severity,attack_vector,exploitability_score,cwe_ids,affected_products,references,related_cves
0,CVE-1999-0253,1997-01-01T05:00:00.000,2026-06-16T21:48:00.503,IIS 3.0 with the iis-fix hotfix installed allo...,7.5,2.0,HIGH,NETWORK,10.0,[],"[microsoft:internet_information_server, micros...","[http://www.securityfocus.com/bid/1814, http:/...",[]
1,CVE-1999-1417,1998-08-23T04:00:00.000,2026-06-16T21:50:23.803,Format string vulnerability in AnswerBook2 (AB...,7.5,2.0,HIGH,NETWORK,10.0,[],[inso:answerbook2],"[http://www.securityfocus.com/archive/1/10383,...",[]
2,CVE-1999-1030,1999-05-19T04:00:00.000,2026-06-16T21:49:33.570,counter.exe 2.70 allows a remote attacker to c...,5.0,2.0,MEDIUM,NETWORK,10.0,[],[behold_software:web_page_counter],[http://marc.info/?l=bugtraq&m=92713790426690&...,[]
3,CVE-2000-0302,2000-03-31T05:00:00.000,2026-06-16T21:51:26.027,Microsoft Index Server allows remote attackers...,5.0,2.0,MEDIUM,NETWORK,10.0,[],[microsoft:index_server],[http://marc.info/?l=bugtraq&m=95453598317340&...,[]
4,CVE-2000-0380,2000-04-26T04:00:00.000,2026-06-16T21:51:35.753,The IOS HTTP service in Cisco routers and swit...,7.1,2.0,HIGH,NETWORK,8.6,[CWE-20],[cisco:ios],[http://archives.neohapsis.com/archives/bugtra...,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,CVE-2009-1535,2009-06-10T14:30:00.170,2026-06-16T23:07:28.580,The WebDAV extension in Microsoft Internet Inf...,7.5,2.0,HIGH,NETWORK,10.0,[CWE-287],[microsoft:internet_information_services],[http://archives.neohapsis.com/archives/fulldi...,[CVE-2009-1122]
96,CVE-2009-2704,2009-08-11T10:30:00.250,2026-06-16T23:10:04.463,CA SiteMinder allows remote attackers to bypas...,4.3,2.0,MEDIUM,NETWORK,8.6,[CWE-264],[sun:j2ee],"[http://i8jesus.com/?p=55, http://i8jesus.com/...",[]
97,CVE-2008-7159,2009-09-10T21:30:00.953,2026-06-16T23:03:43.820,The silc_asn1_encoder function in lib/silcasn1...,5.8,2.0,MEDIUM,NETWORK,8.6,[CWE-134],[silcnet:silc_toolkit],[http://lists.opensuse.org/opensuse-security-a...,[]
98,CVE-2008-7160,2009-09-10T21:30:01.017,2026-06-16T23:03:43.970,The silc_http_server_parse function in lib/sil...,5.8,2.0,MEDIUM,NETWORK,8.6,[CWE-134],[silcnet:silc_toolkit],[http://lists.opensuse.org/opensuse-security-a...,[]


In [34]:
# Análise exploratória de amostras da NVD
# Uma CVE pode corresponder a várias keywords; a consolidação abaixo deduplica pelo CVE ID.

## Análise estatística para orientar o dashboard

Os exemplos abaixo consultam cinco tecnologias em uma janela de três anos. Cada consulta solicita até 2.000 resultados; `totalResults` informa o total da busca antes do filtro local de `last_modified`. Se `api_results_exceed_page` for verdadeiro, a amostra pode estar incompleta e precisa de paginação ou consulta por intervalos menores antes de ser usada como censo. A série temporal conta por `published`. Frequência de keywords e coocorrência são sinais exploratórios, não prova de causalidade nem medida de prevalência de exploração.

Para o dashboard, priorize: volume e severidade no período; tendência mensal; top produtos/versões CPE e CWEs; vetor de ataque e cobertura de CVSS; e uma lista de CVEs novas relacionadas a produtos já monitorados. Para modelos, os campos NVD ajudam a agrupar e ordenar, mas não rotulam exploração. Para prever exploração ou prioridade operacional, junte inventário de ativos e labels externos como CISA KEV/EPSS, valide com divisão temporal e não use informação publicada depois da data de previsão.

In [35]:
from collections import defaultdict
import time

EXAMPLE_KEYWORDS = [
    "Apache",
    "OpenSSL",
    "Microsoft Windows",
    "WordPress",
    "Linux kernel",
]
ANALYSIS_START = (
    pd.Timestamp.now(tz="UTC") - pd.DateOffset(years=3)
).strftime("%Y-%m-%d")
ANALYSIS_END = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d")

keyword_frames = {}
for keyword_index, keyword in enumerate(EXAMPLE_KEYWORDS):
    keyword_frames[keyword] = fetch_cves(
        keyword,
        results_per_page=200,
        max_results=2000,
        start_date=ANALYSIS_START,
        end_date=ANALYSIS_END,
    )

tagged_frames = []
for keyword, frame in keyword_frames.items():
    if not frame.empty:
        tagged = frame.copy()
        tagged["matched_keyword"] = keyword
        tagged_frames.append(tagged)

if tagged_frames:
    keyword_results = pd.concat(tagged_frames, ignore_index=True)
    cve_data = (
        keyword_results.sort_values("last_modified")
        .drop_duplicates(subset="cve_id", keep="last")
        .reset_index(drop=True)
    )
else:
    keyword_results = pd.DataFrame()
    cve_data = pd.DataFrame()

if not cve_data.empty:
    cve_data["published"] = pd.to_datetime(cve_data["published"], utc=True, errors="coerce")
    cve_data["last_modified"] = pd.to_datetime(cve_data["last_modified"], utc=True, errors="coerce")
    cve_data["cvss_score"] = pd.to_numeric(cve_data["cvss_score"], errors="coerce")

    keyword_summary = pd.DataFrame([
        {
            "keyword": keyword,
            "cves_in_sample": len(frame),
            "nvd_total_results": frame.attrs.get("nvd_total_results", len(frame)),
            "api_results_exceed_page": frame.attrs.get("nvd_total_results", len(frame)) > 2000,
            "median_cvss": frame["cvss_score"].median(),
            "critical_cvss_pct": (
                100 * (pd.to_numeric(frame["cvss_score"], errors="coerce").dropna() >= 9).mean()
                if frame["cvss_score"].notna().any() else 0
            ),
            "cvss_coverage_pct": 100 * frame["cvss_score"].notna().mean(),
            "unique_products": len({
                product for products in frame["affected_products"] for product in products
            }),
        }
        for keyword, frame in keyword_frames.items()
    ]).sort_values("cves_in_sample", ascending=False)

    print("Resumo por keyword (resultados limitados ao tamanho da amostra solicitada):")
    display(keyword_summary)

    severity_summary = (
        cve_data["severity"].fillna("Unknown").value_counts()
        .rename_axis("severity").to_frame("cve_count")
    )
    severity_summary["percent"] = 100 * severity_summary["cve_count"] / len(cve_data)
    print("Distribuição de severidade CVSS na amostra deduplicada:")
    display(severity_summary)

    vector_summary = (
        cve_data["attack_vector"].fillna("Unknown").value_counts()
        .rename_axis("attack_vector").to_frame("cve_count")
    )
    vector_summary["percent"] = 100 * vector_summary["cve_count"] / len(cve_data)
    print("Vetor de ataque CVSS (não equivale a exposição dos ativos da organização):")
    display(vector_summary)

    cvss_by_version = (
        cve_data.dropna(subset=["cvss_score"])
        .groupby("cvss_version")["cvss_score"]
        .agg(cve_count="count", median="median", mean="mean")
        .sort_index(ascending=False)
    )
    print("CVSS separado por versão (evita comparar escalas sem contexto):")
    display(cvss_by_version)

    product_counts = (
        cve_data[["cve_id", "affected_products"]]
        .explode("affected_products")
        .dropna(subset=["affected_products"])
        .groupby("affected_products")["cve_id"].nunique()
        .sort_values(ascending=False)
        .head(20)
        .rename("distinct_cves").to_frame()
    )
    print("Produtos CPE associados ao maior número de CVEs (top 20):")
    display(product_counts)

    cwe_counts = (
        cve_data[["cve_id", "cwe_ids"]].explode("cwe_ids")
        .dropna(subset=["cwe_ids"])
        .groupby("cwe_ids")["cve_id"].nunique()
        .sort_values(ascending=False).head(20)
        .rename("distinct_cves").to_frame()
    )
    print("Fraquezas CWE recorrentes (top 20):")
    display(cwe_counts)

    month_counts = (
        cve_data.dropna(subset=["published"])
        .assign(month=lambda data: data["published"].dt.tz_localize(None).dt.to_period("M").astype(str))
        .groupby("month")["cve_id"].nunique().rename("published_cves").to_frame()
    )
    month_counts["rolling_3_month_mean"] = month_counts["published_cves"].rolling(3, min_periods=1).mean()
    print("CVEs publicadas por mês e média móvel de três meses:")
    display(month_counts.tail(36))

    print("Relações temporais candidatas: CVEs publicadas nos últimos 90 dias ligadas a CVEs anteriores por CPE, CWE ou referência explícita.")
    relationship_rows = []
    ordered = cve_data.dropna(subset=["published"]).sort_values("published").tail(500)
    prior_by_feature = defaultdict(list)
    prior_by_id = {}
    for _, current in ordered.iterrows():
        current_id = current["cve_id"]
        features = {
            "product": set(current["affected_products"] or []),
            "cwe": set(current["cwe_ids"] or []),
        }
        candidates = defaultdict(lambda: {"products": set(), "cwes": set(), "explicit": False})
        for product in features["product"]:
            for prior_id in prior_by_feature[("product", product)]:
                candidates[prior_id]["products"].add(product)
        for cwe in features["cwe"]:
            for prior_id in prior_by_feature[("cwe", cwe)]:
                candidates[prior_id]["cwes"].add(cwe)
        for related_id in current["related_cves"] or []:
            normalized_id = related_id.upper()
            if normalized_id in prior_by_id:
                candidates[normalized_id]["explicit"] = True

        ranked = sorted(
            candidates.items(),
            key=lambda item: (
                item[1]["explicit"],
                len(item[1]["products"]) + len(item[1]["cwes"]),
            ),
            reverse=True,
        )[:3]
        for prior_id, evidence in ranked:
            prior = prior_by_id[prior_id]
            days_apart = (current["published"] - prior["published"]).days
            if days_apart <= 90:
                relationship_rows.append({
                    "earlier_cve": prior_id,
                    "newer_cve": current_id,
                    "days_apart": days_apart,
                    "shared_products": sorted(evidence["products"]),
                    "shared_cwes": sorted(evidence["cwes"]),
                    "explicit_cve_reference": evidence["explicit"],
                })

        prior_by_id[current_id.upper()] = current
        for feature_type, feature_values in features.items():
            for feature_value in feature_values:
                prior_by_feature[(feature_type, feature_value)].append(current_id.upper())

    relationships = pd.DataFrame(relationship_rows)
    if not relationships.empty:
        print("Pares com evidência de relação (amostra limitada aos 500 registros mais recentes):")
        display(relationships.sort_values(
            ["explicit_cve_reference", "days_apart"], ascending=[False, True]
        ).head(30))
    else:
        print("Nenhum par relacionado encontrado na amostra; ausência de par não prova ausência de relação.")
else:
    print("Nenhum CVE retornado. Confira conectividade, limite da API, keywords e período.")

ChunkedEncodingError: Response ended prematurely